In [ ]:
import boto3
import pandas as pd
import os
from dotenv import load_dotenv
from io import StringIO

# --- 1. Configuration ---

# Load environment variables from your .env file
load_dotenv()

# AWS credentials from environment variables
aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = os.getenv("AWS_DEFAULT_REGION", "us-east-1") # Uses a default if not set

# S3 bucket and file paths for the resume matching application
# **ACTION REQUIRED**: Update these if your bucket or file paths are different.
bucket_name = "my-resume-data-store"  # A suggested new bucket name for this project
resume_file_key = "raw-data/Resume.csv"
job_desc_file_key = "raw-data/job_title_des.csv"

# --- 2. Initialize S3 Client ---

try:
    s3 = boto3.client(
        "s3",
        aws_access_key_id=aws_access_key,
        aws_secret_access_key=aws_secret_key,
        region_name=aws_region
    )
    print("✅ Successfully initialized S3 client.")
except Exception as e:
    print(f"❌ Error initializing S3 client: {e}")
    s3 = None

# --- 3. Function to Load Dataframes from S3 ---

def load_data_from_s3(s3_client, bucket, resume_key, job_key):
    """
    Loads resume and job description CSV files from S3 into pandas DataFrames.
    """
    if not s3_client:
        print("S3 client not available. Cannot load data.")
        return None, None

    try:
        # Load Resumes CSV
        print(f"🔄 Loading resumes from s3://{bucket}/{resume_key}...")
        resume_obj = s3_client.get_object(Bucket=bucket, Key=resume_key)
        resume_content = resume_obj["Body"].read().decode("utf-8")
        df_resumes = pd.read_csv(StringIO(resume_content))
        print(f"✅ Successfully loaded {len(df_resumes)} resumes.")

        # Load Job Descriptions CSV
        print(f"🔄 Loading job descriptions from s3://{bucket}/{job_key}...")
        job_obj = s3_client.get_object(Bucket=bucket, Key=job_key)
        job_content = job_obj["Body"].read().decode("utf-8")
        df_job_descriptions = pd.read_csv(StringIO(job_content))
        print(f"✅ Successfully loaded {len(df_job_descriptions)} job descriptions.")
        
        return df_resumes, df_job_descriptions

    except s3_client.exceptions.NoSuchKey as e:
        print(f"❌ ERROR: File not found in S3. Please check your bucket and file key.")
        print(f"   Details: {e}")
        return None, None
    except Exception as e:
        print(f"❌ An unexpected error occurred: {e}")
        return None, None

# --- 4. Main Execution ---

if __name__ == "__main__":
    df_resumes, df_job_description = load_data_from_s3(
        s3_client=s3,
        bucket=bucket_name,
        resume_key=resume_file_key,
        job_key=job_desc_file_key
    )

    if df_resumes is not None and df_job_description is not None:
        print("\n--- Data Loading Complete ---")
        
        print("\nFirst 5 rows of the Resumes DataFrame:")
        print(df_resumes.head())
        
        print("\nFirst 5 rows of the Job Descriptions DataFrame:")
        print(df_job_description.head())
        
        print(f"\nTotal Resumes: {len(df_resumes)}")
        print(f"Total Job Descriptions: {len(df_job_description)}")